# Evaluation


## Compare dense and graph/RL retrieval


In [ ]:
import json
import sys
from pathlib import Path

root = Path("..").resolve()
sys.path.insert(0, str(root / "kg_interface"))

from litrag import KnowledgeGraph, RetrievalEnv, LogisticPolicy
from litrag.evaluate import evaluate, dense_baseline, ALWAYS_EXPAND, NEVER_EXPAND

kg = KnowledgeGraph(str(root / "kg_interface" / "data"))
params = json.loads((root / "kg_interface" / "rl_params.json").read_text())
env = RetrievalEnv(kg, max_steps=params["max_steps"], step_cost=params["step_cost"])
eval_set = [q for q in kg.qa if q.get("split") == "dev"] or kg.qa
policy = LogisticPolicy.load(str(root / "kg_interface" / "policy.npz"))

rows = {
    "dense only": dense_baseline(kg, eval_set, k=params["k"]),
    "never expand": evaluate(kg, env, eval_set, NEVER_EXPAND, k=params["k"], link=params["link"]),
    "always expand": evaluate(kg, env, eval_set, ALWAYS_EXPAND, k=params["k"], link=params["link"]),
    "learned RL policy": evaluate(kg, env, eval_set, lambda f: policy.act(f, greedy=True), k=params["k"], link=params["link"]),
}
print("Selected parameters:", params)
print(rows)
